# Phase 6.4: Scikit-learn Estimators, Pipelines, and ColumnTransformer

**Level:** Practitioner  
**Suggested study time:** 120 minutes  
**Phase:** Machine Learning Fundamentals  
**Prerequisite:** Phases 1–5, especially data quality, statistics, and EDA.

**Navigation:** Previous: `03_Imputation_Encoding_Scaling_and_Feature_Engineering.ipynb` · Next: `05_Regression_and_Classification_Metrics.ipynb`

> **Learning principle:** Do not merely run the code. Predict what each step will do, run it, explain the result, change one assumption, and run it again.

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain **Scikit-learn Estimators, Pipelines, and ColumnTransformer** from first principles.
2. Define and distinguish: estimator API, fit, transform, predict, Pipeline, ColumnTransformer, parameter namespace, feature name.
3. Reproduce the central mechanism with a small Python example.
4. State the assumptions and recognize common failure modes.
5. Apply the idea to a new dataset or software problem.
6. Evaluate whether the result is suitable for a real decision.

## 1. Why this matters

Compose preprocessing and estimation into one fitted object that preserves ordering, reproducibility, and train-serving parity.

In professional work, the difficult part is rarely remembering a method name. The difficult part is determining whether the available data, representation, assumptions, evaluation design, and operating constraints make the method appropriate. This lesson therefore connects the mechanism to **correctness**, **interpretation**, and **deployment consequences**.

**Running case:** Train a complete churn pipeline and calculate ROC-AUC.

A useful answer to any technical problem should separate:

- **The estimand or contract:** What exactly are we trying to calculate or guarantee?
- **The evidence:** Which observations, inputs, and assumptions support it?
- **The method:** What transformation or learning procedure is applied?
- **The validation:** How do we know the result generalizes or remains correct?
- **The action:** What decision changes because of the result?

## 2. Mental model

Think of this lesson as a mapping:

```text
Defined inputs
      ↓
Assumptions and representation
      ↓
Transformation / learning rule
      ↓
Output, score, or state change
      ↓
Validation and interpretation
      ↓
Decision, safeguard, or next experiment
```

Ask these questions before coding:

1. What does one input record represent?
2. Which values are observed, derived, learned, or configured?
3. What is the valid domain of every important value?
4. What information must not be used?
5. What result would a naive baseline produce?
6. Which failure is most costly?

## 3. Core vocabulary

        | Concept | Operational meaning |
        |---|---|
        | **estimator API** | A core idea in Scikit-learn Estimators, Pipelines, and ColumnTransformer; understand its definition, assumptions, operational role, and failure modes rather than memorizing its name. |
| **fit** | A core idea in Scikit-learn Estimators, Pipelines, and ColumnTransformer; understand its definition, assumptions, operational role, and failure modes rather than memorizing its name. |
| **transform** | A core idea in Scikit-learn Estimators, Pipelines, and ColumnTransformer; understand its definition, assumptions, operational role, and failure modes rather than memorizing its name. |
| **predict** | A core idea in Scikit-learn Estimators, Pipelines, and ColumnTransformer; understand its definition, assumptions, operational role, and failure modes rather than memorizing its name. |
| **Pipeline** | An ordered composite estimator that learns and applies transformations with a model as one unit. |
| **ColumnTransformer** | A tool for applying different preprocessing pipelines to different feature subsets. |
| **parameter namespace** | A core idea in Scikit-learn Estimators, Pipelines, and ColumnTransformer; understand its definition, assumptions, operational role, and failure modes rather than memorizing its name. |
| **feature name** | An input variable available at the decision point and defined consistently across training and inference. |

        **Study technique:** Cover the right-hand column and explain each term aloud. Then invent a counterexample showing where a careless definition would fail.

## 4. First-principles workflow

        1. Define the future decision and simulate it with the evaluation split.
2. Establish naive and current-process baselines.
3. Fit all learned transformations only on training partitions.
4. Select metrics and thresholds from business costs and constraints.
5. Analyze errors and uncertainty before increasing model complexity.

        ### Before using a library

        For the central operation, write down:

        - the shape and meaning of every input;
        - the expected output shape and unit;
        - the objective or invariant;
        - one hand-calculated example;
        - computational complexity at a high level;
        - one condition under which the method becomes invalid.

        Library fluency is valuable only after the contract is understood.

## 5. Mathematical or formal lens

A pipeline makes cross-validation refit every learned transform within each fold, closing a common leakage path.

### Interpretation discipline

A formula does not establish that its assumptions are true. A successful function call does not establish that its output is meaningful. Always separate:

- **calculation correctness** — did the code implement the formula or contract?
- **statistical validity** — does the design justify the inference?
- **operational validity** — will equivalent inputs exist at decision time?
- **decision validity** — is the output useful under real costs and constraints?

## 6. Environment setup

Run this cell first. It locates the extracted course root, makes outputs reproducible, and creates an artifacts directory. The examples use only bundled data unless the notebook explicitly says otherwise.

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 7. Worked Python example

Read the code once without running it. Predict:

- the important intermediate objects and their shapes/types;
- what output should appear;
- one line that enforces or assumes a contract;
- one change that would cause a failure or misleading result.

Then run the cell and reconcile the actual output with your prediction.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

df = pd.read_csv(DATA_DIR / "customer_churn.csv")
X = df.drop(columns=["customer_id","churn"])
y = df["churn"]
numeric = X.select_dtypes(include="number").columns
categorical = X.select_dtypes(exclude="number").columns

pipeline = Pipeline([
    ("preprocessing", ColumnTransformer([
        ("num", Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]), numeric),
        ("cat", Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                          ("encode",OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])),
    ("model", LogisticRegression(max_iter=1000)),
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=42, stratify=y
)
pipeline.fit(X_train,y_train)
prob = pipeline.predict_proba(X_test)[:,1]
print("ROC-AUC:", roc_auc_score(y_test, prob))

## 8. How to interpret and challenge the result

Use this checklist after execution:

1. **Mechanism:** Explain how the output followed from the input—not merely what the output says.
2. **Scale and units:** Identify units, ranges, shapes, and any implicit normalization.
3. **Assumptions:** Locate at least one assumption in the data generation, split, formula, or API.
4. **Baseline:** Describe the simplest competing method.
5. **Sensitivity:** Change one input, parameter, seed, or threshold and predict the direction of change.
6. **Failure test:** Supply an empty, missing, extreme, duplicated, or unseen input where relevant.
7. **Decision:** State what action the result supports and what additional evidence is still required.

**Expected learning outcome:** You should be able to reconstruct the logic without this notebook and explain why it is or is not appropriate in another context.

## 9. Guided and independent exercises

        1. **Foundation:** Restate `estimator API` in your own words and give one valid and one invalid example.
2. **Foundation:** Modify the worked program so it accepts a small change in input while preserving its contract.
3. **Practitioner:** Recreate the central calculation without copying the example, then test it on at least three cases.
4. **Practitioner:** Identify one assumption behind Scikit-learn Estimators, Pipelines, and ColumnTransformer and design a diagnostic that could reveal its violation.
5. **Advanced:** Compare the demonstrated approach with an alternative; evaluate correctness, complexity, interpretability, and failure modes.
6. **Advanced:** Apply the lesson to a bundled dataset and write a five-sentence conclusion containing evidence, uncertainty, limitation, and action.

        Record each experiment with:

        | Field | What to write |
        |---|---|
        | Question | The specific behaviour or claim being tested |
        | Change | The one variable or assumption you changed |
        | Prediction | What you expected before execution |
        | Observation | What actually happened |
        | Explanation | Why it happened |
        | Decision | What you would do next |

In [ ]:
# PRACTICE WORKSPACE
# 1. Copy only the smallest part of the worked example you need.
# 2. State your hypothesis in a comment.
# 3. Change one thing at a time.
# 4. Add assertions for important invariants.

lesson_title = 'Scikit-learn Estimators, Pipelines, and ColumnTransformer'
hypothesis = "Write your prediction here"
print(lesson_title)
print("Hypothesis:", hypothesis)

# TODO: Implement the foundation exercise.
# TODO: Add at least two edge-case checks.
# TODO: Apply the idea to a bundled dataset.

## 10. Common mistakes and safeguards

        | Common mistake | Professional safeguard |
        |---|---|
        | Preprocessing before splitting | Put learned transforms inside a pipeline and refit within folds. |
| Using accuracy on rare outcomes | Select metrics based on ranking, costs, and capacity. |
| Tuning on the test set | Keep a final holdout or use nested validation. |
| Assuming 0.5 is the right threshold | Choose an operating point from constraints and validate it. |
| Comparing models on different folds | Use an identical evaluation design and uncertainty estimates. |

        Add one mistake you personally made while studying this notebook. Turning an error into a named diagnostic rule is how experience accumulates.

## 11. Knowledge check

        Answer without looking back:

        1. What problem is `estimator API` intended to solve, and what does it not solve?
2. How would you explain the relationship between `fit` and `transform` to a beginner?
3. Which information is learned from data, which is configured, and which must be fixed before evaluation?
4. What evidence would make the worked result untrustworthy?
5. How would the approach behave on missing, duplicated, extreme, or previously unseen inputs?
6. What would you monitor or test before using this idea in a real decision system?

        ### Interview-level extension

        Explain Scikit-learn Estimators, Pipelines, and ColumnTransformer in three layers:

        1. **30 seconds:** problem, core idea, and one limitation;
        2. **3 minutes:** mechanism, assumptions, evaluation, and example;
        3. **15 minutes:** derivation, implementation, alternatives, failure analysis, and production implications.

## 12. Summary and readiness checkpoint

**Central idea:** Compose preprocessing and estimation into one fitted object that preserves ordering, reproducibility, and train-serving parity.

You are ready to continue when you can:

- [ ] define the principal concepts without memorized wording;
- [ ] reproduce the worked mechanism on a small example;
- [ ] explain the formal or mathematical statement;
- [ ] identify assumptions and at least three failure modes;
- [ ] design an appropriate test or evaluation;
- [ ] connect the output to a decision;
- [ ] complete the practitioner exercise without copying.

**Next:** `05_Regression_and_Classification_Metrics.ipynb`